**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/rozsa-2023"], check=True)
    os.chdir("/content/encore/entries/rozsa-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Rozsa et al. (2023) on DANDI:000341, part 3: in vivo recordings

Paper: Rozsa, Toth, Olah, Baka, Lakovics, Barzo and Tamas (2023), Sci. Adv. 9, eade4511,
doi:10.1126/sciadv.ade4511. Claims file: `claims/rozsa-2023.yaml`, written from the version of
record (Europe PMC full text) and its supplement.

Dataset: DANDI:000341, version 0.241220.1312 (doi:10.48324/dandi.000341/0.241220.1312), the only
published version; the paper cites the dandiset without a version.

This notebook covers the folder `In_vivo` (259 files, 383 GB): whole-cell recordings of layer 1
interneurons in mice, most with a layer 5 LFP. Each file is taken as one cell (checked for
repeats below). It was added after the first verification round, which found that the earlier
block of these claims as too large was not justified.

The dandiset carries no AP labels, so AIS-APs and RA-APs can only be separated by reading (b) of
the claims file's header, a two-component Gaussian mixture on threshold and pre-AP slope fitted to
all in vivo APs pooled, RA-APs being the component with the lower threshold (`raf.gmm_classes`).
Its components are checked against the paper's description of RA-APs (lower threshold and lower
slope) before any claim is judged.

| Claim | Here |
|---|---|
| fig4b-invivo-ra-proportion | tested |
| fig4i-invivo-frequencies | tested |
| fig4g-invivo-ra-share | tested |
| fig4f-invivo-pre-ap-fluctuation | tested |
| fig4j-invivo-ais-phase-coupling | tested |
| fig4j-invivo-ra-no-phase | tested |
| fig4-invivo-up-down-balance | tested |
| figs4i-ais-ra-frequency-correlation | tested |
| figs5-not-genotype-or-anesthesia | genotype tested; the condition is not in the files |

In [ ]:
import zlib, time, resource, collections
import numpy as np
import pandas as pd
from scipy import stats, signal
import matplotlib.pyplot as plt
import raf

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
PAPER = "rozsa-2023"


def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())


T0 = time.time()
assets = raf.list_assets()
iv_assets = [a for a in assets if a["path"].startswith("In_vivo/")]
print(f"In_vivo: {len(iv_assets)} files, {sum(a['size'] for a in iv_assets) / 1e9:.0f} GB")
res = raf.process_group(iv_assets, keep=("LFP",), workers=4)
print(f"read in {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Per-cell reduction

APs (main rule, peak above 0 mV; artefacts with a threshold outside -80 to -10 mV or a peak above
100 mV left out) of the Vm sweeps without a current step; the recording duration of a cell is the
summed length of those sweeps. Steady holding current is allowed (the paper's Figure S5F shows
"injected holding currents during recordings"); APs with a positive absolute current at threshold
are left out in a variant.

In [ ]:
def genotype(m):
    d = str(m.get("subject_description", ""))
    return d.split("Animal genotype:")[1].split(",")[0].strip() if "Animal genotype:" in d else "not given"


cells = []
for p, (m, aps, sw, lfp) in res.items():
    vm = sw[sw.chan.str.startswith("Vm")]
    quiet = vm[vm.nsteps == 0]
    a = aps[aps.main & raf.plausible(aps) & aps.series.isin(set(quiet.series))] if len(aps) else aps
    cells.append(dict(file=p.split("/")[-1], subject=m["subject_subject_id"], meta=m, genotype=genotype(m),
                      region=str(m.get("subject_description", "")).split("Brain region:")[-1].strip(),
                      age=m.get("subject_age"), aps=a.assign(file=p.split("/")[-1]) if len(a) else a,
                      n_main=int(aps.main.sum()) if len(aps) else 0, n_vm=len(vm), n_quiet=len(quiet),
                      dur_s=float((quiet.n / quiet.rate).sum()), lfp=lfp, n_lfp=len(lfp),
                      lfp_s=float(sum(len(x) / r for _, r, x in lfp.values())),
                      hold_med=float(quiet.hold.median()) if len(quiet) else np.nan,
                      chans=dict(collections.Counter(sw.chan)),
                      rates=sorted(set(vm.rate.round())),
                      fingerprint=(len(vm), tuple(vm.sort_values("start").vmed.round(3).head(20)))))
print(len(cells), "cells")

## Data check

In [ ]:
C = pd.DataFrame([{k: v for k, v in c.items() if k in ("file", "subject", "genotype", "region", "age", "n_main", "n_vm",
                                                       "n_quiet", "dur_s", "n_lfp", "lfp_s", "hold_med")}
                  | {"n_aps": len(c["aps"])} for c in cells])
print(f"files {len(C)}, subjects {C.subject.nunique()}; Vm sweeps per file median {C.n_vm.median():.0f}; "
      f"files with an LFP {int((C.n_lfp > 0).sum())}")
print("channels (series over all files):", dict(sum((collections.Counter(c["chans"]) for c in cells), collections.Counter())))
print("sampling rates (files):", dict(collections.Counter(r for c in cells for r in c["rates"])))
print(f"recording without steps per cell: median {C.dur_s.median():.0f} s, range {C.dur_s.min():.0f} to {C.dur_s.max():.0f} s; "
      f"LFP per cell median {C.lfp_s.median():.0f} s")
print(f"holding current of the stepless sweeps (median per cell): {C.hold_med.min():.0f} to {C.hold_med.max():.0f} pA, "
      f"median {C.hold_med.median():.0f}")
print(f"APs (peak > 0 mV) {C.n_main.sum()}, kept (stepless sweeps, plausible) {C.n_aps.sum()}; cells with no kept AP "
      f"{int((C.n_aps == 0).sum())}")
print("genotypes (files):", dict(collections.Counter(C.genotype)))
print("regions (files):", dict(collections.Counter(C.region)))
print("ages:", dict(collections.Counter(C.age).most_common(10)))
fp = collections.defaultdict(list)
for c in cells:
    fp[c["fingerprint"]].append(c["file"])
print("files with identical sweep fingerprints:", [v for v in fp.values() if len(v) > 1] or "none")
m0 = cells[0]["meta"]
print("metadata of", cells[0]["file"])
print("   general:", {k: v for k, v in m0["general"].items()})
print("   subject:", {k: v for k, v in m0.items() if k.startswith("subject_")})
print("   electrodes:", m0["electrode_descriptions"])
print("   pharmacology:", m0["pharmacology"])
words = ("anesth", "anaesth", "ketamine", "xylazine", "chloral", "isoflurane", "awake", "urethane")
hits = [c["file"] for c in cells if any(w in str(c["meta"]).lower() for w in words)]
print("files whose metadata mention anesthesia or wakefulness:", hits or "none")
print("session descriptions:", dict(collections.Counter(c["meta"]["session_description"] for c in cells)))
print("experiment descriptions:", dict(collections.Counter(str(c["meta"].get("experiment_description")) for c in cells)))

Clock check for the LFP: the starting times of the Vm and LFP series side by side, and the mean
LFP around the AP peaks against the same average with the AP times shifted by 5 or 10 s.

In [ ]:
for c in [c for c in cells if c["n_lfp"]][:3]:
    sw = res[[p for p in res if p.endswith(c["file"])][0]][2].sort_values("start")
    vm, lf = sw[sw.chan.str.startswith("Vm")], sw[sw.chan.str.startswith("LFP")]
    print(c["file"][:40])
    print("   Vm  (start s, duration s):", [(round(a, 2), round(b, 1)) for a, b in zip(vm.start.head(6), (vm.n / vm.rate).head(6))])
    print("   LFP (start s, duration s):", [(round(a, 2), round(b, 1)) for a, b in zip(lf.start.head(6), (lf.n / lf.rate).head(6))])
    print(f"   Vm series {len(vm)}, LFP series {len(lf)}, sharing a starting time "
          f"{len(set(vm.start.round(3)) & set(lf.start.round(3)))}")
# The same count over every file with LFP series (verifier, round 2).
frac = []
for c in [c for c in cells if c["n_lfp"]]:
    sw = res[[p for p in res if p.endswith(c["file"])][0]][2]
    vm, lf = set(sw[sw.chan.str.startswith("Vm")].start.round(3)), set(sw[sw.chan.str.startswith("LFP")].start.round(3))
    if lf:
        frac.append(len(vm & lf) / len(lf))
print(f"files with LFP series {len(frac)}; LFP series sharing a starting time with a Vm series: median fraction "
      f"{np.median(frac):.3f}, lowest {min(frac):.3f}, files where every one does {sum(f == 1 for f in frac)}")


def ap_triggered_lfp(c, shift=0.0, half=1.0):
    a = c["aps"]
    if not len(a) or not c["lfp"]:
        return None, 0, np.nan
    t = (a.sweep_start + a.t_peak).to_numpy() + shift
    segs = []
    for st, rate, x in c["lfp"].values():
        h = int(half * rate)
        i = np.round((t - st) * rate).astype(int)
        i = i[(i - h >= 0) & (i + h < len(x))]
        for k in i[:2000]:
            seg = x[k - h:k + h + 1].astype(float)
            segs.append(seg - seg.mean())
            used = rate
    if not segs:
        return None, 0, np.nan
    L = min(len(z) for z in segs)
    return np.mean([z[:L] for z in segs], axis=0), len(segs), used


trig = []
for c in cells:
    m0, n0, fs0 = ap_triggered_lfp(c)
    if m0 is None or n0 < 20:
        continue
    c0, w0 = len(m0) // 2, int(round(0.05 * fs0))
    null = [ap_triggered_lfp(c, shift=sh)[0] for sh in (-10, -5, 5, 10)]
    null = [z[len(z) // 2 - w0:len(z) // 2 + w0 + 1].mean() for z in null if z is not None]
    trig.append(dict(file=c["file"][:32], n_aps=n0, lfp_at_ap=m0[c0 - w0:c0 + w0 + 1].mean(),
                     shifted_mean=np.mean(null), shifted_sd=np.std(null)))
trig = pd.DataFrame(trig)
z = (trig.lfp_at_ap - trig.shifted_mean) / trig.shifted_sd.replace(0, np.nan)
print(f"cells with an LFP and at least 20 APs: {len(trig)}; LFP within 50 ms of the AP below its value with the AP "
      f"times shifted in {int((trig.lfp_at_ap < trig.shifted_mean).sum())}; |z| > 2 in {int((z.abs() > 2).sum())}; "
      f"z median {z.median():.2f}")

Reading (b): the mixture on all kept in vivo APs pooled. The component means, SDs and weights are
printed, with whether the RA component (lower threshold) also has the lower slope, as the paper
describes RA-APs.

In [ ]:
A = pd.concat([c["aps"] for c in cells if len(c["aps"])], ignore_index=True)
sd0 = seed("fig4b-invivo-ra-proportion")
A["ra"], GM = raf.gmm_classes(A, sd0)
A["steady"] = A.i_abs > 5
print("mixture components:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in GM.items()})
print("RA component size under seeds +0, +1000 to +4000:", raf.gmm_seed_range(A, sd0))
print(f"APs {len(A)}, RA {int(A.ra.sum())} ({A.ra.mean() * 100:.1f}%); under a positive holding current "
      f"{int(A.steady.sum())}; without a stimulus series {int(A.i_abs.isna().sum())}")
print(A.groupby("ra")[["thr", "slope", "sd_thr", "sd_peak"]].median().round(2))
fig, ax = plt.subplots(figsize=(5, 4))
ax.hexbin(A.thr, A.slope.clip(-5, 15), gridsize=70, bins="log", extent=(-80, -10, -5, 15))
ax.plot([GM["thr_ra"], GM["thr_ais"]], [GM["slope_ra"], GM["slope_ais"]], "r+", ms=14, mew=2)
ax.set(xlabel="threshold (mV)", ylabel="pre-AP slope (mV/ms), clipped to -5 to 15",
       title="in vivo APs; + component means")
plt.show()
print(f"slopes outside -5 to 15 mV/ms: {int((~A.slope.between(-5, 15)).sum())}")
per = []
for c in cells:
    a = A[A.file == c["file"]]
    per.append(dict(file=c["file"], subject=c["subject"], genotype=c["genotype"], dur_s=c["dur_s"],
                    n_ais=int((~a.ra).sum()), n_ra=int(a.ra.sum()),
                    n_ra_nosteady=int((a.ra & ~a.steady).sum()), n_ais_nosteady=int((~a.ra & ~a.steady).sum())))
P = pd.DataFrame(per)
P["f_ais"] = P.n_ais / P.dur_s.where(P.dur_s > 0)
P["f_ra"] = P.n_ra / P.dur_s.where(P.dur_s > 0)
P["n_all"] = P.n_ais + P.n_ra
print(P.describe().round(3))

## fig4b-invivo-ra-proportion

> In vivo, 75 of 242 (31%) mouse layer 1 interneurons fire spontaneous APs from relatively
> hyperpolarized potentials, identified as RA-APs.

In [ ]:
for label, col in (("all kept APs", "n_ra"), ("APs under a positive holding current left out", "n_ra_nosteady")):
    for k in (1, 2):
        n = int((P[col] >= k).sum())
        print(f"{label}, at least {k} RA-AP: {n} of {len(P)} cells = {n / len(P):.3f}; of the {int((P.n_all > 0).sum())} "
              f"cells with any AP = {n / max(1, int((P.n_all > 0).sum())):.3f}")
print("expected 0.31 (75 of 242), tolerance 0.031")

**Result: blocked (methods_underspecified).** The mixture's lower-threshold component has the steeper mean slope (about 3.2 against 2.0 mV/ms), the opposite of the paper's RA-APs, so its classes are not the paper's and reading (a) cannot be run. The mixture is fitted to a seeded draw of 200,000 of the 291,968 APs, and its counts differ slightly between runs, so the numbers below are rounded; the exact ones are printed above. For the record, with these classes about 0.75 of the 259 cells (0.88 of the 220 cells with any AP) have at least one RA-AP, against 0.31.


## fig4i-invivo-frequencies

> In vivo layer 1 interneurons fire sporadically, with AIS-APs at 0.65 Hz and RA-APs at 0.004 Hz,
> AIS-APs being more frequent.

Readings of the criterion: (i) all cells; (ii) cells with RA-APs for the RA-AP value and all cells
for the AIS-AP value; (iii) cells with any APs. Mann-Whitney U test of the per-cell frequencies.

In [ ]:
Q = P[P.dur_s > 0]
rd = {"(i) all cells": (Q.f_ais, Q.f_ra), "(ii) RA from cells with RA-APs": (Q.f_ais, Q.f_ra[Q.n_ra > 0]),
      "(iii) cells with any APs": (Q.f_ais[Q.n_all > 0], Q.f_ra[Q.n_all > 0])}
for k, (fa, fr) in rd.items():
    p = stats.mannwhitneyu(fa, fr).pvalue
    print(f"{k}: AIS-AP median {fa.median():.4f} Hz (n={len(fa)}, SD {fa.std():.3f}), RA-AP median {fr.median():.4f} Hz "
          f"(n={len(fr)}, SD {fr.std():.3f}); Mann-Whitney P = {p:.2g}; AIS higher: {fa.median() > fr.median()}")
print(f"cells without a stepless recording (left out): {int((P.dur_s == 0).sum())}")
print("expected AIS 0.65 Hz (tolerance 0.065), RA 0.004 Hz (tolerance 0.0004), P < 0.001")

**Result: blocked (methods_underspecified).** The mixture's lower-threshold component has the steeper mean slope (about 3.2 against 2.0 mV/ms), the opposite of the paper's RA-APs, so its classes are not the paper's and reading (a) cannot be run. The mixture is fitted to a seeded draw of 200,000 of the 291,968 APs, and its counts differ slightly between runs, so the numbers below are rounded; the exact ones are printed above. For the record, with these classes the AIS-AP median frequency is about 0.36 Hz and the RA-AP median about 0.010 Hz over all 259 cells (Mann-Whitney P below 1e-19), about 0.56 and 0.023 Hz over the 220 cells with APs.


## fig4g-invivo-ra-share

> RA-APs make up 3.4% of all APs recorded in vivo; the RA-AP share ranges from 0 (162 cells) to
> 100% (6 cells), and averages 17% in the 75 cells with RAF.

In [ ]:
share = P.n_ra / P.n_all.where(P.n_all > 0)
raf_cells = share[P.n_ra > 0]
print(f"RA-APs {P.n_ra.sum()} of {P.n_all.sum()} APs = {P.n_ra.sum() / P.n_all.sum() * 100:.2f}%")
print(f"cells with APs and no RA-AP {int(((P.n_all > 0) & (P.n_ra == 0)).sum())}; cells with no AP at all "
      f"{int((P.n_all == 0).sum())}; cells with only RA-APs {int(((P.n_all > 0) & (P.n_ais == 0)).sum())}")
print(f"RA-AP share in cells with RA-APs (n={len(raf_cells)}): mean {raf_cells.mean() * 100:.1f}%, median "
      f"{raf_cells.median() * 100:.1f}%, SD {raf_cells.std() * 100:.1f}%")
print("expected: 3.4% (tolerance 0.34), 162 cells without RA-APs (16.2), 6 with only RA-APs (0.6), 17% (1.7)")

**Result: blocked (methods_underspecified).** The mixture's lower-threshold component has the steeper mean slope (about 3.2 against 2.0 mV/ms), the opposite of the paper's RA-APs, so its classes are not the paper's and reading (a) cannot be run. The mixture is fitted to a seeded draw of 200,000 of the 291,968 APs, and its counts differ slightly between runs, so the numbers below are rounded; the exact ones are printed above. For the record, with these classes RA-APs are about 7.5% of all APs; 26 or 27 cells with APs have none and 9 have only RA-APs; the share in cells with RA-APs is about 16% on average (median about 7%).


## fig4f-invivo-pre-ap-fluctuation

> In vivo, membrane potential fluctuations (SD) 0 to 50 ms before AIS-APs are larger than before
> RA-APs.

As for fig3f in the slices notebook: the SD of the membrane potential in the 50 ms window ending
at the threshold or at the peak (the criterion's readings), APs pooled; per-cell medians and
isolated APs (no AP in the preceding 50 ms) as variants.

In [ ]:
D = A.sort_values(["file", "series", "t_thr"]).copy()
prev = D.groupby(["file", "series"], observed=True).t_peak.shift(1)
D["isolated"] = (D.t_thr - prev).fillna(np.inf) > 0.05
for col in ("sd_thr", "sd_peak"):
    for iso in (False, True):
        d = D[D.isolated] if iso else D
        a, b = d[~d.ra][col].dropna(), d[d.ra][col].dropna()
        p = stats.mannwhitneyu(a, b).pvalue
        pc = d.dropna(subset=[col]).groupby(["file", "ra"])[col].median().unstack()
        pcp = stats.mannwhitneyu(pc[False].dropna(), pc[True].dropna()).pvalue if True in pc and pc[True].notna().sum() else np.nan
        print(f"window to {col[3:]}{' [isolated APs]' if iso else ''}: AIS-AP median {a.median():.2f} mV (n={len(a)}), "
              f"RA-AP {b.median():.2f} mV (n={len(b)}), Mann-Whitney P = {p:.2g}; per-cell medians P = {pcp:.2g}")
print("expected: AIS-AP SD higher, Mann-Whitney P < 0.001")

**Result: blocked (methods_underspecified).** The mixture's lower-threshold component has the steeper mean slope (about 3.2 against 2.0 mV/ms), the opposite of the paper's RA-APs, so its classes are not the paper's and reading (a) cannot be run. The mixture is fitted to a seeded draw of 200,000 of the 291,968 APs, and its counts differ slightly between runs, so the numbers below are rounded; the exact ones are printed above. For the record, with these classes the SD before RA-APs is the higher one (window to threshold about 7.3 against 7.2 mV, P below 1e-14; window to peak about 8.3 against 7.9 mV).


## fig4j-invivo-ais-phase-coupling, fig4j-invivo-ra-no-phase and fig4-invivo-up-down-balance

> In vivo, AIS-APs are coupled to the trough of the slow (0.5 to 4 Hz) LFP oscillation (mean
> vector length 0.019, angle 330 degrees). RA-APs are not significantly coupled to any phase of
> the slow oscillation. AIS-APs dominate during up states and RA-APs during down states.

The Methods' phase procedure, as in the slices notebook: the 4 s LFP segment centred on each AP
peak, FFT power times frequency, the AP kept when the power in 0.5 to 4 Hz over that in 0.5 to
45 Hz exceeds 0.1; the LFP band-passed 0.5 to 4 Hz (Butterworth order 2, zero phase; order 4 the
variant), Hilbert phase at the AP peak shifted by 180 degrees so that the trough is 0 degrees.
All cells with an LFP and all states pooled (criterion).

In [ ]:
FS = 500.0


def ap_phases(c, aps, order=2):
    out = []
    for name, (st, rate, x) in c["lfp"].items():
        dur = len(x) / rate
        t = (aps.sweep_start + aps.t_peak).to_numpy()
        sel = aps[(t >= st + 2) & (t <= st + dur - 2)]
        if not len(sel):
            continue
        q = int(round(rate / FS))
        y = signal.decimate(x.astype(float), q, ftype="fir") if q > 1 else x.astype(float)
        fs = rate / q
        sos = signal.butter(order, (0.5, 4.0), btype="band", fs=fs, output="sos")
        ph = np.angle(signal.hilbert(signal.sosfiltfilt(sos, y)), deg=True)
        for _, ap in sel.iterrows():
            i = int(round((ap.sweep_start + ap.t_peak - st) * fs))
            h = int(round(2 * fs))
            if i - h < 0 or i + h > len(y):
                continue
            seg = y[i - h:i + h]
            F = np.fft.rfftfreq(len(seg), 1 / fs)
            Pw = np.abs(np.fft.rfft(seg - seg.mean())) ** 2 * F
            ratio = Pw[(F >= 0.5) & (F <= 4)].sum() / Pw[(F >= 0.5) & (F <= 45)].sum()
            out.append(dict(file=c["file"], ra=ap.ra, steady=ap.steady, ratio=ratio, phase=(ph[i] + 180) % 360))
    return pd.DataFrame(out)


PH = {}
for order in (2, 4):
    parts = [ap_phases(c, A[A.file == c["file"]], order) for c in cells if c["lfp"] and len(c["aps"])]
    PH[order] = pd.concat([p for p in parts if len(p)])
print({o: len(v) for o, v in PH.items()}, "APs with a 4 s LFP segment, from", PH[2].file.nunique(), "cells")


def phase_report(Pd, label):
    out = {}
    for ra, name in ((False, "AIS-APs"), (True, "RA-APs")):
        d = Pd[Pd.ra == ra]
        k = d[d.ratio > 0.1]
        rr, ang, p, n = raf.rayleigh(k.phase)
        out[name] = (rr, ang, p, n)
        print(f"{label} {name}: with LFP {len(d)} from {d.file.nunique()} cells; kept by delta ratio {n} from "
              f"{k.file.nunique()} cells; vector length {rr:.3f}, angle {ang:.0f} deg, Rayleigh P = {p:.2g}")
    return out


PR = {}
PR["o2"] = phase_report(PH[2], "order 2")
PR["o4"] = phase_report(PH[4], "order 4")
PR["nosteady"] = phase_report(PH[2][~PH[2].steady], "order 2, APs under a positive holding current left out")

In [ ]:
rr, ang, p, n = PR["o2"]["AIS-APs"]
print(f"AIS-APs: n {n}, vector length {rr:.3f} (expected 0.019, tolerance 0.0019), angle {ang:.0f} deg "
      f"(expected 330, distance {raf.circ_dist(ang, 330):.0f}, tolerance 33), Rayleigh P = {p:.2g} (expected < 0.001)")

**Result: blocked (methods_underspecified).** The mixture's lower-threshold component has the steeper mean slope (about 3.2 against 2.0 mV/ms), the opposite of the paper's RA-APs, so its classes are not the paper's and reading (a) cannot be run. The mixture is fitted to a seeded draw of 200,000 of the 291,968 APs, and its counts differ slightly between runs, so the numbers below are rounded; the exact ones are printed above. For the record, with these classes about 224,500 AIS-APs from 199 cells are coupled at 332 degrees (Rayleigh P printed as 0) with a vector length of about 0.43, against 0.019.


In [ ]:
for k in ("o2", "o4", "nosteady"):
    rr, ang, p, n = PR[k]["RA-APs"]
    print(f"{k}: RA-APs n {n}, vector length {rr:.3f}, angle {ang:.0f} deg, Rayleigh P = {p:.3g}")
print("expected: no coupling, Rayleigh P >= 0.05 (paper P = 0.15)")

**Result: blocked (methods_underspecified).** The mixture's lower-threshold component has the steeper mean slope (about 3.2 against 2.0 mV/ms), the opposite of the paper's RA-APs, so its classes are not the paper's and reading (a) cannot be run. The mixture is fitted to a seeded draw of 200,000 of the 291,968 APs, and its counts differ slightly between runs, so the numbers below are rounded; the exact ones are printed above. For the record, with these classes the 17,000 or so RA-APs are coupled too (323 degrees, length about 0.38, Rayleigh P printed as 0).


Up and down parts of the cycle (criterion of fig4-invivo-up-down-balance): (i) halves, up from 270
to 90 degrees through 0, down from 90 to 270; (ii) quarters centred on 0 (up, 315 to 45) and 180
degrees (down, 135 to 225). APs kept by the delta ratio, order 2.

In [ ]:
K = PH[2][PH[2].ratio > 0.1]
for reading, up, down in (("(i) halves", lambda x: (x >= 270) | (x < 90), lambda x: (x >= 90) & (x < 270)),
                          ("(ii) quarters", lambda x: (x >= 315) | (x < 45), lambda x: (x >= 135) & (x < 225))):
    u, d = K[up(K.phase)], K[down(K.phase)]
    ra_u, ais_u, ra_d, ais_d = int(u.ra.sum()), int((~u.ra).sum()), int(d.ra.sum()), int((~d.ra).sum())
    s_u, s_d = ra_u / max(1, ra_u + ais_u), ra_d / max(1, ra_d + ais_d)
    p = stats.fisher_exact([[ra_d, ais_d], [ra_u, ais_u]]).pvalue
    print(f"{reading}: up AIS {ais_u}, RA {ra_u} (RA share {s_u:.3f}); down AIS {ais_d}, RA {ra_d} (RA share {s_d:.3f}); "
          f"RA share higher in down: {s_d > s_u}; AIS outnumber RA in up: {ais_u > ra_u}; Fisher exact P = {p:.2g}")

**Result: blocked (methods_underspecified).** The mixture's lower-threshold component has the steeper mean slope (about 3.2 against 2.0 mV/ms), the opposite of the paper's RA-APs, so its classes are not the paper's and reading (a) cannot be run. The mixture is fitted to a seeded draw of 200,000 of the 291,968 APs, and its counts differ slightly between runs, so the numbers below are rounded; the exact ones are printed above. For the record, with these classes the RA-AP share is higher in the down half of the cycle (about 0.08 against 0.07, Fisher P below 1e-40) and AIS-APs outnumber RA-APs in the up half.


## figs4i-ais-ra-frequency-correlation

> Across in vivo cells, AIS-AP and RA-AP frequencies are weakly positively correlated (R = 0.29).

In [ ]:
for label, d in (("all cells", Q), ("cells with RA-APs", Q[Q.n_ra > 0])):
    r1, p1 = stats.pearsonr(d.f_ais, d.f_ra)
    r2, p2 = stats.spearmanr(d.f_ais, d.f_ra)
    print(f"{label} (n={len(d)}): Pearson R = {r1:.3f} (P = {p1:.2g}); Spearman R = {r2:.3f} (P = {p2:.2g})")
print("expected R = 0.29 (tolerance 0.029), P < 0.001")

**Result: blocked (methods_underspecified).** The mixture's lower-threshold component has the steeper mean slope (about 3.2 against 2.0 mV/ms), the opposite of the paper's RA-APs, so its classes are not the paper's and reading (a) cannot be run. The mixture is fitted to a seeded draw of 200,000 of the 291,968 APs, and its counts differ slightly between runs, so the numbers below are rounded; the exact ones are printed above. For the record, with these classes the per-cell AIS-AP and RA-AP frequencies correlate with Pearson R of about 0.54 and Spearman R of about 0.68 over all 259 cells (P below 1e-19), about 0.51 and 0.61 over the cells with RA-APs.


## figs5-not-genotype-or-anesthesia

> RA-AP occurrence in vivo cannot be explained by mouse genotype or by anesthesia condition.

The genotype is in each file's `subject/description`; no field names the anesthesia condition
(data check). The genotype part is run as the criterion says: Mann-Whitney U tests between every
pair of genotypes, Bonferroni corrected, and a Kruskal-Wallis test.

In [ ]:
G = Q.groupby("genotype")
print(G.agg(cells=("file", "size"), ra_freq_median=("f_ra", "median"), ais_freq_median=("f_ais", "median"),
            with_ra=("n_ra", lambda x: float((x > 0).mean()))).round(4))
groups = {g: d.f_ra.to_numpy() for g, d in G if len(d) >= 2}
names = sorted(groups)
pairs = [(a, b) for i, a in enumerate(names) for b in names[i + 1:]]
for a, b in pairs:
    p = stats.mannwhitneyu(groups[a], groups[b]).pvalue
    print(f"  {a} vs {b}: Mann-Whitney P = {p:.3g}, Bonferroni {min(1, p * len(pairs)):.3g}")
if len(groups) > 1:
    print("Kruskal-Wallis across genotypes: P = %.3g" % stats.kruskal(*groups.values()).pvalue)
print(f"notebook time {(time.time() - T0) / 60:.1f} min, peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

**Result: blocked (metadata_insufficient).** No field of the in vivo files names the anesthesia condition (data check), so the condition part cannot be tested. The genotype is in `subject/description` (five values, including `mouse` and `mouse-black6`, which may be one strain). With the mixture classes, whose RA component has the steeper slope, RA-AP frequency does not differ by genotype (Kruskal-Wallis P near 0.27; smallest Bonferroni-corrected pairwise P near 0.3).

## Summary

| Claim | Outcome |
|---|---|
| fig4b-invivo-ra-proportion | blocked, methods_underspecified |
| fig4i-invivo-frequencies | blocked, methods_underspecified |
| fig4g-invivo-ra-share | blocked, methods_underspecified |
| fig4f-invivo-pre-ap-fluctuation | blocked, methods_underspecified |
| fig4j-invivo-ais-phase-coupling | blocked, methods_underspecified |
| fig4j-invivo-ra-no-phase | blocked, methods_underspecified |
| fig4-invivo-up-down-balance | blocked, methods_underspecified |
| figs4i-ais-ra-frequency-correlation | blocked, methods_underspecified |
| figs5-not-genotype-or-anesthesia | blocked, metadata_insufficient |

How the analysis differs from the paper's. The dandiset has no AP labels. The one runnable reading
of the claims file, a two-component mixture on threshold and pre-AP slope, gives a lower-threshold
component with the steeper slope in these recordings, while the paper describes RA-APs as having
the lower threshold and the lower slope. The claims are therefore blocked; the numbers under the
mixture's classes are shown for each. The claims file's reading (b) does not ask for this check of
the components; applied as written, it would give mixed outcomes (fig4-invivo-up-down-balance
reproduced, fig4i and figs4i partially reproduced, as the verifier noted in round 2). The mixture is
fitted to a seeded draw of 200,000 APs and its counts differ slightly between runs.

What was found about the dataset. The folder holds 259 files from 149 mice against the paper's 242
cells; 245 files have an LFP. In 135 of them every LFP series starts at the same time as a Vm series (median
fraction 1.000; in some files none does), and in 143 of the 167 files with at least 20 APs the
AP-triggered LFP average departs from its time-shifted value (|z| > 2, median z -10.4), which is
consistent with a shared clock. The
genotype is recorded in each file's subject description, the anesthesia condition nowhere. Some
files also hold camera images (`Moovie`).